In [1]:
%pip install xgboost


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")

In [3]:
url = "https://raw.githubusercontent.com/samyuthakanakamedala/mobile-price-prediction/refs/heads/main/smartphones.csv"

df = pd.read_csv(url)

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
display(df.head())

Dataset Shape: (758, 27)

First 5 Rows:


,smartphone_brand,model,price_inr,rating_score,processor_name,processor_brand,core_count,clock_speed_ghz,ram_gb,storage_gb,...,battery_mah,fast_charging,charging_watt,rear_camera_count,front_camera_count,rear_camera_main_mp,front_camera_main_mp,os_name,memory_card_supported,memory_card_type
0,blackzone,BlackZone S25 Ultra Maxx,8999,63,unisoc sc9863a1,unisoc,8,1.6,8,128,...,5000,False,10.0,1,1,32.0,16.0,android,NaN,NaN
1,ringme,Ringme Bold P70,5800,44,mediatek helio,mediatek,2,1.3,4,128,...,3260,False,10.0,1,0,13.0,NaN,android,NaN,NaN
2,lava,Lava Shark 5G (4GB RAM + 128GB),8998,60,unisoc t765,unisoc,8,2.3,4,128,...,5000,True,10.0,1,1,13.0,5.0,android,True,dedicated
3,vivo,Vivo Y19s 5G (4GB RAM + 128GB),11999,69,mediatek dimensity 6300,mediatek,8,2.4,4,128,...,6000,True,15.0,2,1,13.0,5.0,android,True,dedicated
4,vivo,Vivo Y19s 5G (4GB RAM + 64GB),10999,68,mediatek dimensity 6300,mediatek,8,2.4,4,64,...,6000,True,15.0,2,1,13.0,5.0,android,True,dedicated


In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Rows: 758
Columns: 27
Missing values: 419
Duplicate rows: 0


In [5]:
# Remove duplicate rows
df = df.drop_duplicates()

# Remove rows with missing target
df = df.dropna(subset=["price_inr"])

# Identify numerical and categorical columns
num_cols_all = df.select_dtypes(include="number").columns
cat_cols_all = df.select_dtypes(exclude="number").columns

# Fill numerical missing values with median
df[num_cols_all] = df[num_cols_all].fillna(
    df[num_cols_all].median()
)

# Fill categorical missing values with mode
for col in cat_cols_all:
    df[col] = df[col].fillna(df[col].mode()[0])

# Remove invalid prices
df = df[df["price_inr"] > 0]

# Reset index
df = df.reset_index(drop=True)

print("Rows:", len(df))
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

Rows: 758
Missing values: 0
Duplicates: 0


In [6]:
# Total number of cameras
df["total_camera_count"] = (
    df["front_camera_count"] +
    df["rear_camera_count"]
)

# Total screen pixels
df["screen_pixels"] = (
    df["res_width_px"] *
    df["res_height_px"]
)

# Pixels per inch
df["ppi"] = (
    np.sqrt(
        df["res_width_px"]**2 +
        df["res_height_px"]**2
    ) / df["display_inches"]
)

# Charging efficiency
df["charging_efficiency"] = (
    df["charging_watt"] /
    (df["battery_mah"] + 1)
)

# Premium feature count
df["premium_features"] = (
    df["has_5g"].astype(int) +
    df["has_nfc"].astype(int) +
    df["has_ir_blaster"].astype(int) +
    df["fast_charging"].astype(int)
)

print("Feature engineering completed.")

Feature engineering completed.


In [7]:
features = [
    "clock_speed_ghz",
    "storage_gb",
    "ram_gb",
    "rating_score",
    "res_width_px",
    "res_height_px",
    "display_inches",
    "battery_mah",
    "charging_watt",
    "rear_camera_count",
    "front_camera_count",

    # Categorical features
    "smartphone_brand",
    "processor_brand",
    "processor_name",
    "os_name",
    "has_5g",
    "has_nfc",
    "has_ir_blaster",
    "fast_charging",
    "memory_card_supported",
    "memory_card_type",

    # Engineered features
    "total_camera_count",
    "screen_pixels",
    "ppi",
    "charging_efficiency",
    "premium_features"
]

X = df[features]
y = df["price_inr"]

print("Number of features:", len(features))
print("Dataset shape:", X.shape)

Number of features: 26
Dataset shape: (758, 26)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 606
Testing samples: 152


In [9]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numerical features:")
print(num_cols)

print("\nCategorical features:")
print(cat_cols)

Numerical features:
['clock_speed_ghz', 'storage_gb', 'ram_gb', 'rating_score', 'res_width_px', 'res_height_px', 'display_inches', 'battery_mah', 'charging_watt', 'rear_camera_count', 'front_camera_count', 'total_camera_count', 'screen_pixels', 'ppi', 'charging_efficiency', 'premium_features']

Categorical features:
['smartphone_brand', 'processor_brand', 'processor_name', 'os_name', 'has_5g', 'has_nfc', 'has_ir_blaster', 'fast_charging', 'memory_card_supported', 'memory_card_type']


In [10]:
for col in num_cols:

    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    X_train[col] = X_train[col].astype(float).clip(
        lower,
        upper
    )

    X_test[col] = X_test[col].astype(float).clip(
        lower,
        upper
    )

print("Outliers capped successfully.")

Outliers capped successfully.


In [11]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Fit encoder only on training data
X_train_cat = encoder.fit_transform(
    X_train[cat_cols]
)

X_test_cat = encoder.transform(
    X_test[cat_cols]
)

# Standardization
scaler = StandardScaler()

X_train_num = scaler.fit_transform(
    X_train[num_cols]
)

X_test_num = scaler.transform(
    X_test[num_cols]
)

# Combine numerical + categorical
X_train_processed = np.hstack([
    X_train_num,
    X_train_cat
])

X_test_processed = np.hstack([
    X_test_num,
    X_test_cat
])

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)


Training shape: (606, 178)
Testing shape: (152, 178)


In [12]:
rf_model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_processed,
    y_train
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [13]:
gb_model = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(
    X_train_processed,
    y_train
)

print("Gradient Boosting trained successfully.")

Gradient Boosting trained successfully.


In [14]:
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror",
    n_jobs=-1
)

xgb_model.fit(
    X_train_processed,
    y_train
)

print("XGBoost trained successfully.")

XGBoost trained successfully.


In [15]:
models = {
    "Random Forest": rf_model,
    "Gradient Boosting": gb_model,
    "XGBoost": xgb_model
}

results = []

for name, model in models.items():

    predictions = model.predict(X_test_processed)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

print(
    results_df.round(2).to_string(index=False)
)

            Model     MAE    RMSE   R2
    Random Forest 3244.04 7869.92 0.90
Gradient Boosting 3185.71 7507.64 0.91
          XGBoost 3593.11 8658.62 0.88


In [16]:
display(
    results_df.style
    .format({
        "MAE": "₹{:,.2f}",
        "RMSE": "₹{:,.2f}",
        "R2": "{:.4f}"
    })
)

,Model,MAE,RMSE,R2
0,Random Forest,"₹3,244.04","₹7,869.92",0.9010
1,Gradient Boosting,"₹3,185.71","₹7,507.64",0.9099
2,XGBoost,"₹3,593.11","₹8,658.62",0.8801


In [17]:
final_model = gb_model
final_model_name = "Gradient Boosting"

print("FINAL MODEL:", final_model_name)

FINAL MODEL: Gradient Boosting


In [19]:
final_result = results_df[
    results_df["Model"] == final_model_name
].iloc[0]

print("Final Model:", final_model_name)
print("MAE :", round(final_result["MAE"], 2))
print("RMSE:", round(final_result["RMSE"], 2))
print("R²  :", round(final_result["R2"], 2))

Final Model: Gradient Boosting
MAE : 3185.71
RMSE: 7507.64
R²  : 0.91


In [20]:
import joblib

# Save Random Forest
joblib.dump(
    rf_model,
    "random_forest_model.pkl"
)

# Save Gradient Boosting
joblib.dump(
    gb_model,
    "gradient_boosting_model.pkl"
)

# Save XGBoost
joblib.dump(
    xgb_model,
    "xgboost_model.pkl"
)

# Save final model
joblib.dump(
    final_model,
    "mobile_price_final_model.pkl"
)

# Save encoder
joblib.dump(
    encoder,
    "mobile_price_encoder.pkl"
)

# Save scaler
joblib.dump(
    scaler,
    "mobile_price_scaler.pkl"
)

print("All models and preprocessing files saved successfully.")

All models and preprocessing files saved successfully.


In [21]:
import json

metrics_data = {}

for _, row in results_df.iterrows():

    metrics_data[row["Model"]] = {
        "MAE": float(row["MAE"]),
        "RMSE": float(row["RMSE"]),
        "R2": round(float(row["R2"]), 2)
    }

metrics_data["Final Model"] = final_model_name

with open("model_metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=4)

print(json.dumps(metrics_data, indent=4))

{
    "Random Forest": {
        "MAE": 3244.0422491019212,
        "RMSE": 7869.918438069494,
        "R2": 0.9
    },
    "Gradient Boosting": {
        "MAE": 3185.7084650611564,
        "RMSE": 7507.635568458461,
        "R2": 0.91
    },
    "XGBoost": {
        "MAE": 3593.1055908203125,
        "RMSE": 8658.617212549007,
        "R2": 0.88
    },
    "Final Model": "Gradient Boosting"
}


In [22]:
feature_schema = {
    "features": features,
    "numerical_features": num_cols,
    "categorical_features": cat_cols
}

with open("feature_schema.json", "w") as f:
    json.dump(feature_schema, f, indent=4)

print("Feature schema saved successfully.")

Feature schema saved successfully.


In [23]:
def prepare_prediction_input(phone):

    phone = phone.copy()

    # Feature engineering
    phone["total_camera_count"] = (
        phone["front_camera_count"] +
        phone["rear_camera_count"]
    )

    phone["screen_pixels"] = (
        phone["res_width_px"] *
        phone["res_height_px"]
    )

    phone["ppi"] = (
        np.sqrt(
            phone["res_width_px"] ** 2 +
            phone["res_height_px"] ** 2
        )
        / phone["display_inches"]
    )

    phone["charging_efficiency"] = (
        phone["charging_watt"] /
        (phone["battery_mah"] + 1)
    )

    phone["premium_features"] = (
        phone["has_5g"].astype(int) +
        phone["has_nfc"].astype(int) +
        phone["has_ir_blaster"].astype(int) +
        phone["fast_charging"].astype(int)
    )

    # Keep exact feature order
    phone = phone[features]

    # Numerical processing
    phone_num = scaler.transform(
        phone[num_cols]
    )

    # Categorical processing
    phone_cat = encoder.transform(
        phone[cat_cols]
    )

    # Combine
    phone_processed = np.hstack([
        phone_num,
        phone_cat
    ])

    return phone_processed

In [24]:
test_phone = X.iloc[[0]].copy()

print("Test phone details:")
display(test_phone)

Test phone details:


,clock_speed_ghz,storage_gb,ram_gb,rating_score,res_width_px,res_height_px,display_inches,battery_mah,charging_watt,rear_camera_count,...,has_nfc,has_ir_blaster,fast_charging,memory_card_supported,memory_card_type,total_camera_count,screen_pixels,ppi,charging_efficiency,premium_features
0,1.6,128,8,63,720,1612,6.6,5000,10.0,1,...,False,False,False,True,dedicated,2,1160640,267.498015,0.002,0


In [25]:
test_processed = prepare_prediction_input(
    test_phone
)

predicted_price = final_model.predict(
    test_processed
)[0]

print("Final Model:", final_model_name)
print(
    "Predicted Mobile Price: ₹",
    round(predicted_price, 2)
)

Final Model: Gradient Boosting
Predicted Mobile Price: ₹ 9893.7


In [26]:
custom_phone = pd.DataFrame([{
    "clock_speed_ghz": 2.8,
    "storage_gb": 256,
    "ram_gb": 8,
    "rating_score": 80,
    "res_width_px": 2400,
    "res_height_px": 1080,
    "display_inches": 6.7,
    "battery_mah": 5000,
    "charging_watt": 67,
    "rear_camera_count": 3,
    "front_camera_count": 1,

    "smartphone_brand": df["smartphone_brand"].mode()[0],
    "processor_brand": df["processor_brand"].mode()[0],
    "processor_name": df["processor_name"].mode()[0],
    "os_name": df["os_name"].mode()[0],

    "has_5g": True,
    "has_nfc": True,
    "has_ir_blaster": False,
    "fast_charging": True,

    "memory_card_supported": False,
    "memory_card_type": df["memory_card_type"].mode()[0]
}])

display(custom_phone)

,clock_speed_ghz,storage_gb,ram_gb,rating_score,res_width_px,res_height_px,display_inches,battery_mah,charging_watt,rear_camera_count,...,smartphone_brand,processor_brand,processor_name,os_name,has_5g,has_nfc,has_ir_blaster,fast_charging,memory_card_supported,memory_card_type
0,2.8,256,8,80,2400,1080,6.7,5000,67,3,...,realme,mediatek,mediatek dimensity 6300,android,True,True,False,True,False,dedicated


In [27]:
custom_processed = prepare_prediction_input(
    custom_phone
)

custom_prediction = final_model.predict(
    custom_processed
)[0]

print("FINAL MODEL:", final_model_name)
print(
    "Estimated Mobile Price: ₹",
    round(custom_prediction, 2)
)

FINAL MODEL: Gradient Boosting
Estimated Mobile Price: ₹ 24909.52
